# EDA - card_identity

Goals:
- Count images and ground-truth files
- Check matching basenames
- Inspect annotation schemas
- Sample quads and note image metadata


In [ ]:
import json
import os
import glob
import shutil
from pathlib import Path


In [ ]:
ROOT = Path.cwd()
if not (ROOT / 'card_identity').exists() and (ROOT.parent / 'card_identity').exists():
    ROOT = ROOT.parent

IMG_DIR = ROOT / 'card_identity' / '01_alb_id' / 'images'
GT_DIR = ROOT / 'card_identity' / '01_alb_id' / 'ground_truth'

img_paths = glob.glob(str(IMG_DIR / '**' / '*.tif'), recursive=True)
gt_paths = glob.glob(str(GT_DIR / '**' / '*.json'), recursive=True)

print('images:', len(img_paths))
print('ground_truth json:', len(gt_paths))


In [ ]:
def base_name(p):
    return os.path.splitext(os.path.basename(p))[0]

img_bases = {base_name(p) for p in img_paths}
gt_bases = {base_name(p) for p in gt_paths}

print('image bases:', len(img_bases))
print('json bases:', len(gt_bases))
print('missing images for json:', len(gt_bases - img_bases))
print('missing json for images:', len(img_bases - gt_bases))


In [ ]:
schema_counts = {}
for p in gt_paths:
    with open(p, encoding='utf-8') as f:
        data = json.load(f)
    if isinstance(data, dict):
        keys = tuple(sorted(data.keys()))
    else:
        keys = ('__non_dict__',)
    schema_counts[keys] = schema_counts.get(keys, 0) + 1

print('schema variants:', len(schema_counts))
for k, v in schema_counts.items():
    print(v, k)


In [ ]:
# Optional: inspect one annotation file
sample = gt_paths[0]
with open(sample, encoding='utf-8') as f:
    data = json.load(f)
print('sample file:', sample)
print('keys:', list(data.keys()) if isinstance(data, dict) else type(data))


In [ ]:
# Optional: image metadata if tiffinfo is available
tiffinfo = shutil.which('tiffinfo')
if tiffinfo:
    sample_img = img_paths[0]
    print('tiffinfo sample:', sample_img)
    os.system(f"{tiffinfo} '{sample_img}' | head -n 12")
else:
    print('tiffinfo not available')
